
# Cardivore_AI — Playground (Ingest Mode, Rich Output, MySQL Load)

This notebook:
1. Loads Market Movers CSVs from `data/raw/`
2. Cleans currency and normalizes names
3. Merges PSA10 + Raw datasets
4. Computes `total_cost`, `profit_margin`, `roi_ratio`
5. Builds `search_string` + `ebay_search_url`
6. Saves **all rows** to `data/processed/all_card_data.csv`
7. Loads **all rows** into MySQL table `card_sales_data` (overwrites existing data)
8. Shows a pretty **Rich** table of sample rows

> **Note**: This notebook intentionally does **no filtering**. All filtering happens later in analysis.


In [ ]:
# --- Path Setup ---
import sys, os, pandas as pd
from rich.console import Console
from rich.table import Table
from IPython.display import display

# Add the outer project root to the Python path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

console = Console()
console.print(f"🧠 Added to sys.path: [bold cyan]{PROJECT_ROOT}[/bold cyan]")

# Import helpers (these now work)
from cardivore_ai.utils.cards import build_search_string, build_ebay_search_url
from cardivore_ai.utils.io_utils import clean_prices
from cardivore_ai.utils.timers import timeit
from cardivore_ai.utils.db_utils import insert_dataframe, get_mysql_connection

# --- Data Paths ---
DATA_DIR = os.path.abspath(os.path.join(PROJECT_ROOT, "data"))
RAW_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")

PSA10_FILE = os.path.join(RAW_DIR, "market-movers-export-20251022PSA10.csv")
RAW_FILE   = os.path.join(RAW_DIR, "market-movers-export-20251022_raw.csv")

console.print(f"📂 PSA10 file path: [bold green]{PSA10_FILE}[/bold green]")
console.print(f"📂 RAW file path:   [bold green]{RAW_FILE}[/bold green]")


In [ ]:
@timeit
def load_data():
    psa10 = pd.read_csv(PSA10_FILE)
    raw = pd.read_csv(RAW_FILE)
    console.print(f"📈 PSA10 rows: {len(psa10):,} | RAW rows: {len(raw):,}")

    # Clean price columns
    psa10["Avg"] = clean_prices(psa10["Avg"])
    raw["Avg"]   = clean_prices(raw["Avg"])

    # Derive comparable "card_base_name" for merge
    psa10["card_base_name"] = psa10["Card"].astype(str).str[:-7]
    raw["card_base_name"]   = raw["Card"].astype(str).str[:-20]
    return psa10, raw

psa10, raw = load_data()
pd.set_option('display.max_colwidth', None)
display(psa10["card_base_name"].head(),raw["card_base_name"].head(),"Data loaded and displayed above.")


In [ ]:
@timeit
def merge_and_filter(psa10, raw):
    merged = pd.merge(psa10, raw, on="card_base_name", how="inner", suffixes=("_psa10", "_raw"))

    # Rename for clarity
    merged.rename(columns={"Avg_psa10": "psa10_avg", "Avg_raw": "raw_avg"}, inplace=True)

    # Add total cost (includes $25 grading & shipping)
    merged["total_cost"] = merged["raw_avg"] + 25

    # Calculate margins and ROI
    merged["profit_margin"] = merged["psa10_avg"] - merged["total_cost"]
    merged["roi_ratio"] = merged["psa10_avg"] / merged["total_cost"]

    # Keep only decent data (volume > 10 PSA10 sales)
    merged["# of Sales_psa10"] = pd.to_numeric(merged["# of Sales_psa10"], errors="coerce")
    filtered = merged[
        (merged["# of Sales_psa10"] > 0.5) &
        (merged["raw_avg"] <= 500) &
        (merged["roi_ratio"] >= 0.5)
    ].copy()

    # Build search strings + URLs
    filtered["search_string"] = filtered["Card_psa10"].apply(build_search_string)
    filtered["ebay_url"] = filtered["search_string"].apply(build_ebay_search_url)

    return filtered.sort_values(by="roi_ratio", ascending=False)

profitable = merge_and_filter(psa10, raw)
console.print(f"🎯 Profitable cards found: [bold green]{len(profitable):,}[/bold green]")


In [ ]:
psa10_names = set(psa10["card_base_name"].str.strip().str.lower())
raw_names = set(raw["card_base_name"].str.strip().str.lower())

print("Overlap:", len(psa10_names & raw_names))
print("Unique to PSA10:", len(psa10_names - raw_names))
print("Unique to Raw:", len(raw_names - psa10_names))

In [ ]:
psa10_names = set(psa10["card_base_name"].str.strip().str.lower())
raw_names = set(raw["card_base_name"].str.strip().str.lower())

print("Overlap:", len(psa10_names & raw_names))
print("Unique to PSA10:", len(psa10_names - raw_names))
print("Unique to Raw:", len(raw_names - psa10_names))


In [ ]:
OUTPUT_FILE = os.path.join(PROCESSED_DIR, "profitable_cards1.xlsx")

# Convert HYPERLINK formulas
profitable["eBay Link"] = profitable["ebay_url"].apply(
    lambda url: f'=HYPERLINK("{url}", "eBay Search")'
)

profitable.to_excel(OUTPUT_FILE, index=False, engine="openpyxl")
console.print(f"💾 Excel file saved to: [bold yellow]{OUTPUT_FILE}[/bold yellow]")
